In [0]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [0]:
# =========================
# Paths
# =========================
pred_path = "/Volumes/workspace/default/ather/predictions/"
ranking_path = "/Volumes/workspace/default/ather/model_ranking.csv"

In [0]:
# =========================
# Load Predictions
# =========================
base_pred_df = pd.read_csv(pred_path + "predictions.csv")
lstm_ens_df = pd.read_csv(pred_path + "lstm_ensemble_predictions.csv")

In [0]:
# =========================
# Extract Predictions
# =========================
y_true = base_pred_df["y_true"].values

predictions = {
    "LinearRegression": base_pred_df["LinearRegression"].values,
    "RandomForest": base_pred_df["RandomForest"].values,
    "XGBoost": base_pred_df["XGBoost"].values,
    "AdaBoost": base_pred_df["AdaBoost"].values,
    "DecisionTree": base_pred_df["DecisionTree"].values,
    "LSTM": lstm_ens_df["LSTM"].values,
    "Ensemble": lstm_ens_df["Ensemble"].values
}

In [0]:
# =========================
# Align lengths
# =========================
min_len = min([len(y_true)] + [len(v) for v in predictions.values()])
y_true = y_true[:min_len]

for k in predictions:
    predictions[k] = predictions[k][:min_len]

In [0]:
# =========================
# Compute Metrics
# =========================
results = []

for model_name, y_pred in predictions.items():
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    results.append({
        "Model": model_name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

In [0]:
# =========================
# Create Ranking Table
# =========================
ranking_df = pd.DataFrame(results)
ranking_df = ranking_df.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

In [0]:
# =========================
# Extract Top 2 Models
# =========================
top_model_1 = ranking_df.iloc[0]["Model"]
top_model_2 = ranking_df.iloc[1]["Model"]

print("Top 1 Model:", top_model_1)
print("Top 2 Model:", top_model_2)

In [0]:
# =========================
# Save Ranking Table
# =========================
ranking_df.to_csv(ranking_path, index=False)